# Création de la base SQLite — Plato's Pizza

Objectif : lire les 4 fichiers CSV du dossier `data/` et les charger dans une base `pizza.db` relationnelle, avec les bonnes clés primaires et étrangères.

Tables à créer :
- `pizza_types` (pizza_type_id, name, category, ingredients)
- `pizzas` (pizza_id, pizza_type_id [FK], size, price)
- `orders` (order_id, date, time)
- `order_details` (order_details_id, order_id [FK], pizza_id [FK], quantity)

Ce notebook doit pouvoir être réexécuté entièrement pour reconstruire la base à l'identique (reproductibilité du livrable).

## 1. Imports

In [1]:
# pandas : pour lire les CSV et les envoyer vers la base SQL
import pandas as pd
# sqlite3 : pour créer et piloter la base de données locale
import sqlite3

## 2. Lecture des fichiers CSV

In [2]:
# Chargement des 4 fichiers sources dans des DataFrames pandas
df_orders = pd.read_csv("data/orders.csv")
df_pizzas = pd.read_csv("data/pizzas.csv")
# encoding='latin-1' : ce fichier contient un caractère spécial non reconnu en UTF-8 par défaut
df_pizza_types = pd.read_csv("data/pizza_types.csv", encoding='latin-1')
df_order_details = pd.read_csv("data/order_details.csv")

## 3. Contrôle qualité des données (avant insertion)

In [3]:
# Valeurs manquantes : compte le nombre de cellules vides par colonne, pour chaque table
print("Valeurs manquantes :")
print(df_pizza_types.isna().sum())
print(df_pizzas.isna().sum())
print(df_orders.isna().sum())
print(df_order_details.isna().sum())

# Doublons : compte le nombre de lignes strictement identiques dans chaque table
print("\nDoublons :")
print("pizza_types:", df_pizza_types.duplicated().sum())
print("pizzas:", df_pizzas.duplicated().sum())
print("orders:", df_orders.duplicated().sum())
print("order_details:", df_order_details.duplicated().sum())

# Types de colonnes : vérifie que price est bien numérique, quantity bien un entier, etc.
print("\nTypes de colonnes :")
print(df_pizzas.dtypes)
print(df_order_details.dtypes)

# Dates : vérifie que la colonne date est bien convertible en date (sinon pd.to_datetime lèverait une erreur)
pd.to_datetime(df_orders["date"])
print("\nColonne date : conversion en date réussie, pas de valeur invalide.")

Valeurs manquantes :
pizza_type_id    0
name             0
category         0
ingredients      0
dtype: int64
pizza_id         0
pizza_type_id    0
size             0
price            0
dtype: int64
order_id    0
date        0
time        0
dtype: int64
order_details_id    0
order_id            0
pizza_id            0
quantity            0
dtype: int64

Doublons :
pizza_types: 0
pizzas: 0
orders: 0
order_details: 0

Types de colonnes :
pizza_id             str
pizza_type_id        str
size                 str
price            float64
dtype: object
order_details_id    int64
order_id            int64
pizza_id              str
quantity            int64
dtype: object

Colonne date : conversion en date réussie, pas de valeur invalide.


## 4. Connexion / création de la base SQLite

In [4]:
# Ouvre (ou crée si besoin) le fichier pizza.db et retourne une connexion active
conn = sqlite3.connect("pizza.db")

## 5. Création des tables (schéma relationnel)

In [5]:
# Table des types de pizzas (catalogue des recettes)
conn.execute("""
CREATE TABLE pizza_types (
    pizza_type_id TEXT PRIMARY KEY,
    name TEXT,
    category TEXT,
    ingredients TEXT
)
""")

# Table des pizzas (déclinaisons taille/prix de chaque type)
conn.execute("""
CREATE TABLE pizzas (
    pizza_id TEXT PRIMARY KEY,
    pizza_type_id TEXT,
    size TEXT,
    price REAL,
    FOREIGN KEY (pizza_type_id) REFERENCES pizza_types(pizza_type_id)
)
""")

# Table des commandes (une ligne par commande passée)
conn.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    date TEXT,
    time TEXT
)
""")

# Table des lignes de commande (quelles pizzas, en quelle quantité, pour quelle commande)
conn.execute("""
CREATE TABLE order_details (
    order_details_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    pizza_id TEXT,
    quantity INTEGER,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (pizza_id) REFERENCES pizzas(pizza_id)
)
""")


## 6. Insertion des données dans les tables

In [6]:
# Envoi de chaque DataFrame vers sa table SQL correspondante
# if_exists="append" : ajoute les lignes dans la table déjà créée (ne la recrée pas)
# index=False : ne pas insérer l'index pandas comme colonne, les tables ont déjà leurs propres clés
# L'ordre respecte les dépendances : pizza_types avant pizzas, orders avant order_details
df_pizza_types.to_sql("pizza_types", conn, if_exists="append", index=False)
conn.commit()
df_pizzas.to_sql("pizzas", conn, if_exists="append", index=False)
conn.commit()
df_orders.to_sql("orders", conn, if_exists="append", index=False)
conn.commit()
df_order_details.to_sql("order_details", conn, if_exists="append", index=False)
conn.commit()

## 7. Vérifications rapides (nombre de lignes, aperçu)

In [7]:
# Compte le nombre de lignes de chaque table directement depuis la base (pas depuis les DataFrames)
# pour prouver que les données sont bien arrivées dans pizza.db
print(pd.read_sql("SELECT COUNT(*) FROM pizza_types", conn))
print(pd.read_sql("SELECT COUNT(*) FROM pizzas", conn))
print(pd.read_sql("SELECT COUNT(*) FROM orders", conn))
print(pd.read_sql("SELECT COUNT(*) FROM order_details", conn))

   COUNT(*)
0        32
   COUNT(*)
0        96
   COUNT(*)
0     21350
   COUNT(*)
0     48620
